# G3: repository contributor concentration (`03_repository_contributor_concentration`)

**Question.** For each repository and day: how many people did the engineering work, and how much of it came from the most active ones?

This notebook builds the Gold table `github_lakehouse.gold.repository_contributor_concentration`.

Contract: `docs/gold_contracts.md`, section 11. The way a Gold table is refreshed is in section 6.

| | |
|---|---|
| Reads | Six Silver tables in `github_lakehouse.silver`: push, pull request, issue, issue comment, pull request review and pull request review comment events |
| Writes | `gold.repository_contributor_concentration`, its staging copy `gold_staging.repository_contributor_concentration`, and one row in `audit.gold_pipeline_runs` |
| Grain | One row per repository per UTC day on which it had at least one event in the six tables |
| Key | `repo_id`, `activity_date` |

The notebook has the same eleven steps as `06_repository_activity_profile`, and it places the Silver tables under each other in the same way. What is new: it counts in **two levels**. First the events of each person, then, from those, the repository-day. And it **ranks** the people of a repository-day to find the most active ones.

## Six kinds of engineering events

| Kind | Silver table |
|---|---|
| Push | `push_events` |
| Pull request | `pull_request_events` |
| Issue | `issue_events` |
| Issue comment | `issue_comment_events` |
| Review | `pull_request_review_events` |
| Inline review comment | `pull_request_review_comment_events` |

Watch, fork and release events are not engineering work. This notebook does not read them at all.

## The rules of this table

- **One event is one row** of the six tables, and it is counted once.
- **An event is attributable when its actor is known.** An event without an actor is counted as unattributable. It is never given to a made-up person.
- **Every share uses the attributable events as its base**, not all events. With 5 events of one person, 3 of another and 2 without an actor, the top share is 5 of 8, not 5 of 10.
- **The top actor** is the person with the most events of the repository on that day. **The top 3** are the three people with the most events, together. With three people or fewer, the top 3 are everybody.
- **The HHI** (Herfindahl index) is one number for the concentration: the sum of the squared shares of all people, with the shares as fractions. It is 1 when one person did everything and gets closer to 0 as the work spreads over more people.
- **A day without any attributable event** has 0 people, and its top counts, shares and HHI are NULL: there is nobody to rank.
- **Every actor is kept, bots included** (decision 6).
- **Actor IDs are not published.** When two people have the same number of events, it does not matter which of them is ranked first: the numbers in the table stay the same.

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | Create the table | `CREATE TABLE IF NOT EXISTS` with the columns of the contract |
| 3 | Affected days | Decides which days this run rebuilds |
| 4 | Read Silver | Places the six tables under each other, only the affected days |
| 5 | Build | The counts per repository and day, the events per person, the ranking, the top counts, the HHI and the name label |
| 6 | Stage | Writes the result to the staging copy |
| 7 | Quality gates | Checks the staging copy and the event IDs |
| 8 | Publish | Replaces the affected days in the Gold table, only if the gates passed |
| 9 | Quality numbers | Totals for the audit row |
| 10 | Audit | Writes one audit row. Stops with an error if the run failed |
| 11 | Results | Shows the audit row and the Gold table |

Before the first run, `08_gold_audit_table` must have run once.

## 1. Parameters

The notebook has the three widgets every Gold notebook has.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | All days are rebuilt and the whole Gold table is replaced | `YYYY-MM-DD`: only the days touched by that source date are rebuilt and replaced |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>` |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_repository_contributor_concentration"

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
    processing_scope = "ALL_DATES"
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

    processing_scope = "SINGLE_DATE"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The table names are set **only** in this cell: the schema of the six Silver tables, the Gold table, its staging copy and the audit table. Later cells never change them, otherwise a test run could write to production.

The six Silver tables are always in one schema. So the schema is chosen first, and the six names are built from it.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    SILVER_SCHEMA = "github_lakehouse.silver"
    GOLD_TABLE = "github_lakehouse.gold.repository_contributor_concentration"
    STAGING_TABLE = "github_lakehouse.gold_staging.repository_contributor_concentration"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    SILVER_SCHEMA = f"github_lakehouse.{test_schema}"
    GOLD_TABLE = f"github_lakehouse.{test_schema}.repository_contributor_concentration"
    STAGING_TABLE = f"github_lakehouse.{test_schema}.repository_contributor_concentration_staging"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

# The six Silver tables of engineering work.
PUSH_TABLE = SILVER_SCHEMA + ".push_events"
PULL_REQUEST_TABLE = SILVER_SCHEMA + ".pull_request_events"
ISSUE_TABLE = SILVER_SCHEMA + ".issue_events"
ISSUE_COMMENT_TABLE = SILVER_SCHEMA + ".issue_comment_events"
REVIEW_TABLE = SILVER_SCHEMA + ".pull_request_review_events"
REVIEW_COMMENT_TABLE = SILVER_SCHEMA + ".pull_request_review_comment_events"

print("Silver schema :", SILVER_SCHEMA)
print("Gold table    :", GOLD_TABLE)
print("Staging table :", STAGING_TABLE)
print("Audit table   :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. The ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 Is the setup done?

The audit table is created by `08_gold_audit_table`. The six Silver tables are created by the Silver notebooks. If one of the seven tables does not exist, the notebook stops here with a clear message.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

silver_tables = [
    PUSH_TABLE,
    PULL_REQUEST_TABLE,
    ISSUE_TABLE,
    ISSUE_COMMENT_TABLE,
    REVIEW_TABLE,
    REVIEW_COMMENT_TABLE,
]

for silver_table in silver_tables:
    if not spark.catalog.tableExists(silver_table):
        raise RuntimeError(
            f"{silver_table} does not exist. Run the Silver notebook of this table first."
        )

print("Setup found  :", AUDIT_TABLE)
print("Sources found:")
for silver_table in silver_tables:
    print("  ", silver_table)

## 2. Create the Gold table

The columns, their types and their order are those of the contract. `CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, so this step is safe to run again.

| Column | Meaning |
|---|---|
| `repo_id` | Key |
| `activity_date` | Key. The UTC date of the events |
| `repo_name` | Label: the latest name seen that day in the six tables |
| `engineering_activity_events` | All events of the six kinds that day |
| `attributable_events` | Events with a known actor |
| `unattributable_events` | Events without a known actor. The two add up to all events |
| `actor_coverage_pct` | 100 × attributable events ÷ all events |
| `unique_active_actors` | Different actors that day. 0 when no event is attributable |
| `top_actor_events` | Events of the most active actor |
| `top_actor_share_pct` | 100 × top actor events ÷ attributable events |
| `top_3_actor_events` | Events of the three most active actors together |
| `top_3_actor_share_pct` | 100 × top 3 actor events ÷ attributable events |
| `hhi_actor_activity` | Concentration index from 0 to 1, with 4 decimals |
| `_gold_run_id` | The run that last wrote the row |
| `_gold_updated_at` | When that run wrote it |

The last five value columns are NULL when the day has no attributable event.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        -- Key and label
        repo_id                       BIGINT,
        activity_date                 DATE,
        repo_name                     STRING,

        -- All events, and who did them
        engineering_activity_events   BIGINT,
        attributable_events           BIGINT,
        unattributable_events         BIGINT,
        actor_coverage_pct            DOUBLE,
        unique_active_actors          BIGINT,

        -- The most active people
        top_actor_events              BIGINT,
        top_actor_share_pct           DOUBLE,
        top_3_actor_events            BIGINT,
        top_3_actor_share_pct         DOUBLE,
        hhi_actor_activity            DOUBLE,

        -- Technical columns
        _gold_run_id                  STRING,
        _gold_updated_at              TIMESTAMP
    )
    USING DELTA
""")

print("Table ready:", GOLD_TABLE)

## 3. Affected days

A Gold row is a summary of a whole day. When one new event arrives for a day, the row of that day has to be calculated again from **all** events of that day. So the first question of every run is: which days does this run have to rebuild?

### 3.1 The Silver rows in the scope of this run

- With a `source_date`, the scope is the Silver rows that were loaded from that source date.
- With a blank date, the scope is all Silver rows.

This table reads six Silver tables, so the scope is taken from those six. `UNION ALL` places the results of the six queries under each other. Only the event date is needed here. A new watch, fork or release event does not change this table, so those tables are not looked at.

A row without an event date cannot be placed on a day. It is left out and counted here, and the count goes into the audit row. The profiling run found 0 such rows.

In [0]:
if source_date_value is None:
    scope_filter = "true"
else:
    scope_filter = f"_source_date = DATE'{source_date}'"

scope_rows = spark.sql(f"""
    SELECT event_date FROM {PUSH_TABLE}           WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {PULL_REQUEST_TABLE}   WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {ISSUE_TABLE}          WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {ISSUE_COMMENT_TABLE}  WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {REVIEW_TABLE}         WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {REVIEW_COMMENT_TABLE} WHERE {scope_filter}
""")
scope_rows.createOrReplaceTempView("scope_rows")

scope_row = spark.sql("""
    SELECT
        COUNT(*)                      AS rows_in_scope,
        count_if(event_date IS NULL)  AS rows_without_event_date
    FROM scope_rows
""").collect()[0]

rows_in_scope = scope_row["rows_in_scope"]
rows_without_event_date = scope_row["rows_without_event_date"]

print("Scope filter                :", scope_filter)
print("Silver rows in the scope    :", rows_in_scope)
print("Rows without an event date  :", rows_without_event_date)

### 3.2 The days these rows belong to

The affected days are the event dates of the rows in the scope, from any of the six tables. A day is rebuilt also when only one of the six tables has a new row for it.

In the week that is loaded, the event date and the source date of a row are always the same day (measured by the profiling run). So a run for one source date rebuilds exactly one day.

In [0]:
day_rows = spark.sql("""
    SELECT DISTINCT event_date
    FROM scope_rows
    WHERE event_date IS NOT NULL
    ORDER BY event_date
""").collect()

affected_dates = []
for day_row in day_rows:
    affected_dates.append(day_row["event_date"])

print("Affected days:", len(affected_dates))
for affected_date in affected_dates:
    print("  ", affected_date)

### 3.3 The day list as SQL text

The later steps need the affected days inside a SQL `IN (...)` list, for example `DATE'2024-02-01', DATE'2024-02-02'`. The same text is used to read Silver, to check the result and to publish, so all three work on exactly the same days.

When there is no affected day, the text is `NULL`. `IN (NULL)` is never true, so the later queries simply find no row.

In [0]:
if len(affected_dates) == 0:
    affected_days_sql = "NULL"
else:
    day_texts = []
    for affected_date in affected_dates:
        day_texts.append("DATE'" + str(affected_date) + "'")
    affected_days_sql = ", ".join(day_texts)

print("Day list:", affected_days_sql)

## 4. Read Silver

All Silver rows of the affected days are read, **not only the rows of this source date**. A row that arrived earlier for the same day must be counted again, otherwise the new Gold row would be too small.

The six tables are placed under each other with `UNION ALL`. They are **not joined**: a join of raw events from two tables would multiply the rows.

`UNION ALL` matches the columns by their position, not by their name. So the six queries select the same six columns in the same order:

| Column | Where it comes from |
|---|---|
| `event_id` | Silver. Used to check that no event is counted twice |
| `event_timestamp` | Silver. Used to find the latest name of the day |
| `activity_date` | Silver `event_date`, with its Gold name |
| `repo_id` | Silver. The key together with the day |
| `repo_name` | Silver. The label |
| `actor_id` | Silver. The person who did the work |

The first of the six queries gives the columns their names. The other five only have to select the same columns in the same order.

Spark does not read anything yet. A temporary view is only a name for a query. The work is done in step 6, when the result is written.

In [0]:
engineering_rows = spark.sql(f"""
    SELECT event_id, event_timestamp, event_date AS activity_date, repo_id, repo_name, actor_id
    FROM {PUSH_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {PULL_REQUEST_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {ISSUE_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {ISSUE_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {REVIEW_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {REVIEW_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})
""")
engineering_rows.createOrReplaceTempView("engineering_rows")

### 4.1 How many Silver rows these days have

The rows are counted directly on each Silver table, not on the view above. So these numbers do not depend on the steps that follow.

The query below has six small queries in brackets. Each one counts the rows of one Silver table and gives one number. Together they make one row with six columns.

The six numbers go into the audit row, one number per table. A quality gate compares their sum with the sum of `engineering_activity_events`: every Silver row of the affected days must be counted exactly once.

In [0]:
count_row = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {PUSH_TABLE}           WHERE event_date IN ({affected_days_sql})) AS push_rows,
        (SELECT COUNT(*) FROM {PULL_REQUEST_TABLE}   WHERE event_date IN ({affected_days_sql})) AS pull_request_rows,
        (SELECT COUNT(*) FROM {ISSUE_TABLE}          WHERE event_date IN ({affected_days_sql})) AS issue_rows,
        (SELECT COUNT(*) FROM {ISSUE_COMMENT_TABLE}  WHERE event_date IN ({affected_days_sql})) AS issue_comment_rows,
        (SELECT COUNT(*) FROM {REVIEW_TABLE}         WHERE event_date IN ({affected_days_sql})) AS review_rows,
        (SELECT COUNT(*) FROM {REVIEW_COMMENT_TABLE} WHERE event_date IN ({affected_days_sql})) AS review_comment_rows
""").collect()[0]

# One number per Silver table. The names are the names of the Silver tables.
silver_row_counts = {
    "push_events": count_row["push_rows"],
    "pull_request_events": count_row["pull_request_rows"],
    "issue_events": count_row["issue_rows"],
    "issue_comment_events": count_row["issue_comment_rows"],
    "pull_request_review_events": count_row["review_rows"],
    "pull_request_review_comment_events": count_row["review_comment_rows"],
}

silver_row_count = sum(silver_row_counts.values())

print("Silver rows on the affected days, per table:")
for silver_table_name in silver_row_counts:
    print("  ", silver_table_name, ":", silver_row_counts[silver_table_name])
print("All six together:", silver_row_count)

## 5. Build

The Gold rows are built in five small queries. Each one is saved as a temporary view.

| Query | View | One row per |
|---|---|---|
| 5.1 | `day_counts` | Repository and day: all events, who did them, and how many people |
| 5.2 | `actor_counts` | Repository, day and person: the events of that person |
| 5.3 | `top_actors` | Repository and day: the top counts and the sum for the HHI, from 5.2 |
| 5.4 | `repo_names` | Repository and day: the name label |
| 5.5 | `gold_rows` | Repository and day: everything together, with the shares and the HHI |

### 5.1 The counts per repository and day

One `GROUP BY` over the stacked events gives one row per repository and day.

- `COUNT(*)` counts every event of the repository on that day.
- `COUNT(actor_id)` counts only the events where the actor is known: the attributable events.
- `count_if(actor_id IS NULL)` counts the events without an actor. The two add up to all events.
- `COUNT(DISTINCT actor_id)` counts the different people and leaves NULL out. With no known actor it gives 0.

**People.** `COUNT(DISTINCT actor_id)` runs over all six kinds of events at once. That is why the tables were stacked first: someone who pushes and reviews on the same day is one person.

In [0]:
day_counts = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        COUNT(*)                     AS engineering_activity_events,
        COUNT(actor_id)              AS attributable_events,
        count_if(actor_id IS NULL)   AS unattributable_events,
        COUNT(DISTINCT actor_id)     AS unique_active_actors
    FROM engineering_rows
    GROUP BY repo_id, activity_date
""")
day_counts.createOrReplaceTempView("day_counts")

### 5.2 The events of each person

This is the first of the two levels: one row per repository, day and person, with the number of events of that person.

Only attributable events are used. `WHERE actor_id IS NOT NULL` leaves the events without an actor out, so they never become a made-up person.

In [0]:
actor_counts = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        actor_id,
        COUNT(*) AS actor_events
    FROM engineering_rows
    WHERE actor_id IS NOT NULL
    GROUP BY repo_id, activity_date, actor_id
""")
actor_counts.createOrReplaceTempView("actor_counts")

### 5.3 The top counts and the sum for the HHI

This is the second level: from the people of step 5.2 back to one row per repository and day.

**The ranking.** `ROW_NUMBER` numbers the people of each repository and day, the most active first. The inner query adds this number as `actor_rank`.

- When two people have the same number of events, the smaller `actor_id` gets the smaller number. This makes the result the same on every run. Which of the two is first does not change any number in the table.

**The values.**

- `MAX(actor_events)` is the number of events of the most active person.
- The top 3 add up the events of the people with the numbers 1, 2 and 3. `CASE` gives 0 for everybody else.
- `SUM(actor_events * actor_events)` adds up the square of each person's events. The HHI in step 5.5 needs it.

**Why the squares.** The HHI is the sum of the squared shares. The share of a person is their events ÷ the attributable events. Squared, that is their events² ÷ attributable². All people have the same divisor, so the HHI is (the sum of all events²) ÷ attributable². Example: three people with 6, 3 and 1 events give (36 + 9 + 1) ÷ 10² = 46 ÷ 100 = 0.46.

A repository-day without any attributable event has no person in step 5.2, so it gets no row here. Step 5.5 handles that.

In [0]:
top_actors = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        MAX(actor_events)                                       AS top_actor_events,
        SUM(CASE WHEN actor_rank <= 3 THEN actor_events ELSE 0 END)  AS top_3_actor_events,
        SUM(actor_events * actor_events)                        AS sum_of_squared_events
    FROM (
        SELECT
            repo_id,
            activity_date,
            actor_events,
            ROW_NUMBER() OVER (
                PARTITION BY repo_id, activity_date
                ORDER BY actor_events DESC, actor_id
            ) AS actor_rank
        FROM actor_counts
    ) ranked_actors
    GROUP BY repo_id, activity_date
""")
top_actors.createOrReplaceTempView("top_actors")

### 5.4 The `repo_name` label

A repository can be renamed, so `repo_name` is only a label and never a key. The label of a day is the name on the **latest** event of that repository on that day, from the six tables. A later watch, fork or release event does not count, because this table does not read them.

`ROW_NUMBER` numbers the events of each repository and day, the latest first. Number 1 is the one we keep.

- Events without a name are left out first, so a missing name is skipped.
- If two events have the same time, the one with the larger `event_id` gets number 1. `event_id` is text and is compared as text. This makes the result the same on every run.

In [0]:
repo_names = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        repo_name
    FROM (
        SELECT
            repo_id,
            activity_date,
            repo_name,
            ROW_NUMBER() OVER (
                PARTITION BY repo_id, activity_date
                ORDER BY event_timestamp DESC, event_id DESC
            ) AS name_rank
        FROM engineering_rows
        WHERE repo_name IS NOT NULL
    ) ranked_names
    WHERE name_rank = 1
""")
repo_names.createOrReplaceTempView("repo_names")

### 5.5 The Gold rows

The counts, the top counts and the label are put together. The columns are in the order of the Gold table.

- `LEFT JOIN` keeps every repository-day of step 5.1. A day without any attributable event finds no row in `top_actors`, and a day whose events all have no name finds no row in `repo_names`. Their columns are then NULL.
- **Coverage:** 100 × attributable events ÷ all events, rounded to 2 decimals. A row always has at least one event, so the coverage is never NULL. A coverage of 0 means: the day has events, but none of them has a known actor.
- **Shares:** 100 × top events ÷ attributable events, rounded to 2 decimals. In SQL, `/` gives a decimal number also when both sides are whole numbers.
- **HHI:** the sum of the squared events ÷ (attributable events × attributable events), rounded to 4 decimals.
- The `CASE` keeps the query from dividing by zero. Without attributable events there is no share and no HHI, so they are NULL. The top counts are NULL in that case too, because there is nobody to rank.
- Every value is calculated from the exact counts and rounded only once, at the end.
- The two technical columns say which run wrote the row and when.

In [0]:
gold_rows = spark.sql(f"""
    SELECT
        c.repo_id,
        c.activity_date,
        n.repo_name,

        c.engineering_activity_events,
        c.attributable_events,
        c.unattributable_events,
        CASE WHEN c.engineering_activity_events > 0
             THEN ROUND(100 * c.attributable_events / c.engineering_activity_events, 2)
        END AS actor_coverage_pct,
        c.unique_active_actors,

        t.top_actor_events,
        CASE WHEN c.attributable_events > 0
             THEN ROUND(100 * t.top_actor_events / c.attributable_events, 2)
        END AS top_actor_share_pct,
        t.top_3_actor_events,
        CASE WHEN c.attributable_events > 0
             THEN ROUND(100 * t.top_3_actor_events / c.attributable_events, 2)
        END AS top_3_actor_share_pct,
        CASE WHEN c.attributable_events > 0
             THEN ROUND(t.sum_of_squared_events / (c.attributable_events * c.attributable_events), 4)
        END AS hhi_actor_activity,

        '{run_id}'           AS _gold_run_id,
        current_timestamp()  AS _gold_updated_at

    FROM day_counts c
    LEFT JOIN top_actors t
      ON  t.repo_id = c.repo_id
      AND t.activity_date = c.activity_date
    LEFT JOIN repo_names n
      ON  n.repo_id = c.repo_id
      AND n.activity_date = c.activity_date
""")
gold_rows.createOrReplaceTempView("gold_rows")

## 6. Stage

The result is written to the staging copy first. The Gold table is not touched yet.

Spark does its work only now, when the rows are written. If something goes wrong, the error is kept in `stage_error`, so that the audit row at the end can report it.

In [0]:
stage_error = None

try:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {STAGING_TABLE} AS
        SELECT * FROM gold_rows
    """)
    print("Staged:", STAGING_TABLE)

except Exception as error:
    stage_error = str(error)[:1000]
    print("Staging FAILED:", stage_error)

## 7. Quality gates

The checks run on the staging copy. One of them, the event IDs, runs on the stacked Silver rows. If a gate fails, nothing is published.

| Gate | It fails when |
|---|---|
| `KEY_NOT_UNIQUE` | A repository and day appear more than once |
| `KEY_IS_NULL` | A row has no `repo_id` or no `activity_date` |
| `COUNT_IS_NULL` | One of the four count columns is missing: all events, attributable, unattributable or people. A count is 0 when nothing was counted, never NULL |
| `ATTRIBUTION_DOES_NOT_ADD_UP` | Attributable + unattributable events is not all events |
| `ACTOR_COUNTS_OUT_OF_ORDER` | There are more people than attributable events, or a row has attributable events but 0 people |
| `TOP_COUNTS_OUT_OF_ORDER` | For a row with people: the top actor has fewer than 1 event or more than the top 3, the top 3 have more than the attributable events, the top actor has fewer events than the average person, or a row with at most 3 people has top 3 events that are not all attributable events |
| `CONCENTRATION_DOES_NOT_FIT_ITS_BASE` | A row with attributable events has no top count, share or HHI, or a row without attributable events has one |
| `SHARE_OUT_OF_RANGE` | A top share is below 0 or above 100 |
| `HHI_OUT_OF_RANGE` | The HHI is above 1, or below 1 ÷ the number of people by more than the rounding (0.0001) |
| `COVERAGE_OUT_OF_RANGE` | A coverage is below 0 or above 100 |
| `COVERAGE_DOES_NOT_FIT_ITS_BASE` | A row with events has no coverage, or a row without events has one |
| `EVENT_ID_NOT_UNIQUE` | An `event_id` appears more than once in the six Silver tables together, on the affected days |
| `EVENT_SUM_DIFFERS_FROM_SILVER` | The sum of `engineering_activity_events` is not the number of rows of the six Silver tables on the affected days |

### 7.1 Keys that appear more than once

In [0]:
duplicate_key_count = 0

if stage_error is None:
    duplicate_key_count = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT repo_id, activity_date
            FROM {STAGING_TABLE}
            GROUP BY repo_id, activity_date
            HAVING COUNT(*) > 1
        ) keys_seen_twice
    """).collect()[0]["duplicate_keys"]

print("Keys that appear more than once:", duplicate_key_count)

### 7.2 Event IDs that appear more than once

Each Silver table has every `event_id` only once. Here the six tables are stacked, so the same check is made on all six together. If one event were in two tables, it would be counted twice.

This check reads the stacked Silver rows of step 4, not the staging copy, because the Gold rows no longer carry event IDs. The profiling run found no `event_id` twice in the nine Silver tables together.

In [0]:
duplicate_event_id_count = 0

if stage_error is None:
    duplicate_event_id_count = spark.sql("""
        SELECT COUNT(*) AS duplicate_event_ids
        FROM (
            SELECT event_id
            FROM engineering_rows
            GROUP BY event_id
            HAVING COUNT(*) > 1
        ) event_ids_seen_twice
    """).collect()[0]["duplicate_event_ids"]

print("Event IDs that appear more than once:", duplicate_event_id_count)

### 7.3 The other checks

One query counts, for every rule, the rows that break it. The expected value of each count is 0. The last two values are the row count and the sum of all events.

- **The top actor and the average.** The most active person has at least as many events as the average person: top × people ≥ attributable events. Example: 10 events among 4 people are 2.5 on average, so the top person has at least 3.
- **The HHI and the number of people.** With n people the HHI is at least 1 ÷ n, when all have the same number of events. The HHI is rounded to 4 decimals, so it can be up to 0.00005 below that. The check allows 0.0001.
- The `CASE` keeps the HHI check from dividing by zero: a row with 0 people has no HHI and no lower limit.
- `COALESCE(SUM(...), 0)`: the sum of no rows is NULL. For the comparison with Silver it has to be 0.

In [0]:
failed_checks = []
staged_row_count = 0
staged_event_sum = 0

if stage_error is None:
    gate_row = spark.sql(f"""
        SELECT
            count_if(repo_id IS NULL OR activity_date IS NULL)
                AS rows_without_key,

            count_if(engineering_activity_events IS NULL
                  OR attributable_events IS NULL
                  OR unattributable_events IS NULL
                  OR unique_active_actors IS NULL)
                AS rows_with_missing_count,

            count_if(attributable_events + unattributable_events <> engineering_activity_events)
                AS rows_where_attribution_does_not_add_up,

            count_if(unique_active_actors > attributable_events
                  OR (attributable_events > 0 AND unique_active_actors = 0))
                AS rows_with_actor_counts_out_of_order,

            count_if(unique_active_actors > 0
                 AND (top_actor_events < 1
                   OR top_actor_events > top_3_actor_events
                   OR top_3_actor_events > attributable_events
                   OR top_actor_events * unique_active_actors < attributable_events
                   OR (unique_active_actors <= 3 AND top_3_actor_events <> attributable_events)))
                AS rows_with_top_counts_out_of_order,

            count_if((attributable_events > 0
                      AND (top_actor_events IS NULL
                        OR top_actor_share_pct IS NULL
                        OR top_3_actor_events IS NULL
                        OR top_3_actor_share_pct IS NULL
                        OR hhi_actor_activity IS NULL))
                  OR (attributable_events = 0
                      AND (top_actor_events IS NOT NULL
                        OR top_actor_share_pct IS NOT NULL
                        OR top_3_actor_events IS NOT NULL
                        OR top_3_actor_share_pct IS NOT NULL
                        OR hhi_actor_activity IS NOT NULL)))
                AS rows_where_concentration_does_not_fit,

            count_if(top_actor_share_pct < 0 OR top_actor_share_pct > 100
                  OR top_3_actor_share_pct < 0 OR top_3_actor_share_pct > 100)
                AS rows_with_share_out_of_range,

            count_if(hhi_actor_activity > 1
                  OR hhi_actor_activity < CASE WHEN unique_active_actors > 0
                                               THEN 1 / unique_active_actors - 0.0001
                                          END)
                AS rows_with_hhi_out_of_range,

            count_if(actor_coverage_pct < 0
                  OR actor_coverage_pct > 100)
                AS rows_with_coverage_out_of_range,

            count_if((engineering_activity_events > 0 AND actor_coverage_pct IS NULL)
                  OR (engineering_activity_events = 0 AND actor_coverage_pct IS NOT NULL))
                AS rows_where_coverage_does_not_fit,

            COUNT(*)                                          AS staged_rows,
            COALESCE(SUM(engineering_activity_events), 0)     AS staged_event_sum

        FROM {STAGING_TABLE}
    """).collect()[0]

    staged_row_count = gate_row["staged_rows"]
    staged_event_sum = gate_row["staged_event_sum"]

    if duplicate_key_count > 0:
        failed_checks.append("KEY_NOT_UNIQUE")
    if gate_row["rows_without_key"] > 0:
        failed_checks.append("KEY_IS_NULL")
    if gate_row["rows_with_missing_count"] > 0:
        failed_checks.append("COUNT_IS_NULL")
    if gate_row["rows_where_attribution_does_not_add_up"] > 0:
        failed_checks.append("ATTRIBUTION_DOES_NOT_ADD_UP")
    if gate_row["rows_with_actor_counts_out_of_order"] > 0:
        failed_checks.append("ACTOR_COUNTS_OUT_OF_ORDER")
    if gate_row["rows_with_top_counts_out_of_order"] > 0:
        failed_checks.append("TOP_COUNTS_OUT_OF_ORDER")
    if gate_row["rows_where_concentration_does_not_fit"] > 0:
        failed_checks.append("CONCENTRATION_DOES_NOT_FIT_ITS_BASE")
    if gate_row["rows_with_share_out_of_range"] > 0:
        failed_checks.append("SHARE_OUT_OF_RANGE")
    if gate_row["rows_with_hhi_out_of_range"] > 0:
        failed_checks.append("HHI_OUT_OF_RANGE")
    if gate_row["rows_with_coverage_out_of_range"] > 0:
        failed_checks.append("COVERAGE_OUT_OF_RANGE")
    if gate_row["rows_where_coverage_does_not_fit"] > 0:
        failed_checks.append("COVERAGE_DOES_NOT_FIT_ITS_BASE")
    if duplicate_event_id_count > 0:
        failed_checks.append("EVENT_ID_NOT_UNIQUE")
    if staged_event_sum != silver_row_count:
        failed_checks.append("EVENT_SUM_DIFFERS_FROM_SILVER")

print("Staged rows (repository-days):", staged_row_count)
print("Sum of engineering_activity_events:", staged_event_sum)
print("Silver rows of the six tables     :", silver_row_count)

if len(failed_checks) == 0:
    print("Quality gates: no gate failed")
else:
    print("Quality gates FAILED:", "; ".join(failed_checks))

## 8. Publish

What happens with the staged rows:

| Situation | Status | Gold table |
|---|---|---|
| Staging failed | `FAILED` | Unchanged |
| There is no affected day, because Silver has no rows in the scope of this run | `NO_INPUT` | Unchanged |
| A quality gate failed | `FAILED` | Unchanged |
| Everything is fine | `SUCCEEDED` | The affected days are replaced by the staged rows |

How the days are replaced:

- **With a `source_date`:** `INSERT INTO ... REPLACE WHERE activity_date IN (...)` deletes the rows of the affected days and inserts the staged rows. The other days are not touched. Delta also refuses a staged row that belongs to another day.
- **With a blank date:** `INSERT OVERWRITE` replaces the whole table. This also removes a day that no longer exists in Silver.

Both are one Delta commit. A reader sees either the old rows or the new ones, never a half-written day. A repository-day that no longer has events disappears, because its whole day is replaced.

In [0]:
run_status = None
error_message = None
published_rows = 0

if stage_error is not None:
    run_status = "FAILED"
    error_message = "Staging failed: " + stage_error

elif len(affected_dates) == 0:
    run_status = "NO_INPUT"

elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Quality gates failed: " + "; ".join(failed_checks)

else:
    # The first line of the statement depends on the scope of the run.
    if source_date_value is None:
        publish_start = f"INSERT OVERWRITE TABLE {GOLD_TABLE}"
    else:
        publish_start = (
            f"INSERT INTO {GOLD_TABLE} "
            f"REPLACE WHERE activity_date IN ({affected_days_sql})"
        )

    print("Publish statement starts with:", publish_start)

    try:
        spark.sql(f"""
            {publish_start}
            SELECT
                repo_id,
                activity_date,
                repo_name,
                engineering_activity_events,
                attributable_events,
                unattributable_events,
                actor_coverage_pct,
                unique_active_actors,
                top_actor_events,
                top_actor_share_pct,
                top_3_actor_events,
                top_3_actor_share_pct,
                hhi_actor_activity,
                _gold_run_id,
                _gold_updated_at
            FROM {STAGING_TABLE}
        """)
        run_status = "SUCCEEDED"
        published_rows = staged_row_count

    except Exception as error:
        run_status = "FAILED"
        error_message = "Publish failed: " + str(error)[:1000]

print("Status:", run_status)
print("Published rows:", published_rows)

## 9. Quality numbers for the audit row

These totals describe what this run calculated. They are counted on the staging copy, so they are there also when the run was not published. None of them stops the run.

After a run for all dates they can be compared directly with the expected values in section 11 of the contract.

- `unique_active_actors` summed over the rows counts a person once for every repository and day on which they worked. It is a number of person-repository-days, not of people.
- `rows_without_attributable_events` counts the rows whose top counts, shares and HHI are NULL.
- `single_actor_rows` counts the rows with exactly one person. Their top share is 100 and their HHI is 1 by definition.

In [0]:
quality_metrics = {
    "repository_days": 0,
    "engineering_activity_events": 0,
    "attributable_events": 0,
    "unattributable_events": 0,
    "unique_active_actors": 0,
    "top_actor_events": 0,
    "top_3_actor_events": 0,
    "rows_without_attributable_events": 0,
    "single_actor_rows": 0,
}

if stage_error is None:
    total_row = spark.sql(f"""
        SELECT
            COUNT(*)                                          AS repository_days,
            COALESCE(SUM(engineering_activity_events), 0)     AS engineering_activity_events,
            COALESCE(SUM(attributable_events), 0)             AS attributable_events,
            COALESCE(SUM(unattributable_events), 0)           AS unattributable_events,
            COALESCE(SUM(unique_active_actors), 0)            AS unique_active_actors,
            COALESCE(SUM(top_actor_events), 0)                AS top_actor_events,
            COALESCE(SUM(top_3_actor_events), 0)              AS top_3_actor_events,
            count_if(attributable_events = 0)                 AS rows_without_attributable_events,
            count_if(unique_active_actors = 1)                AS single_actor_rows
        FROM {STAGING_TABLE}
    """).collect()[0]

    quality_metrics["repository_days"] = total_row["repository_days"]
    quality_metrics["engineering_activity_events"] = total_row["engineering_activity_events"]
    quality_metrics["attributable_events"] = total_row["attributable_events"]
    quality_metrics["unattributable_events"] = total_row["unattributable_events"]
    quality_metrics["unique_active_actors"] = total_row["unique_active_actors"]
    quality_metrics["top_actor_events"] = total_row["top_actor_events"]
    quality_metrics["top_3_actor_events"] = total_row["top_3_actor_events"]
    quality_metrics["rows_without_attributable_events"] = total_row["rows_without_attributable_events"]
    quality_metrics["single_actor_rows"] = total_row["single_actor_rows"]

print("Repository-days                   :", quality_metrics["repository_days"])
print("Engineering events                :", quality_metrics["engineering_activity_events"])
print("  attributable                    :", quality_metrics["attributable_events"])
print("  unattributable                  :", quality_metrics["unattributable_events"])
print("People (person-repository-days)   :", quality_metrics["unique_active_actors"])
print("Events of the top actors          :", quality_metrics["top_actor_events"])
print("Events of the top 3 actors        :", quality_metrics["top_3_actor_events"])
print("Rows without attributable events  :", quality_metrics["rows_without_attributable_events"])
print("Rows with exactly one person      :", quality_metrics["single_actor_rows"])

## 10. Audit

One row is added to the Gold audit table for this run. The values are written in the order of the table's columns.

`source_rows_read` has one number per Silver table: the dictionary of step 4.1.

If the run failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

In [0]:
# The Delta version of the Gold table after this run.
gold_table_version = spark.sql(
    f"DESCRIBE HISTORY {GOLD_TABLE} LIMIT 1"
).collect()[0]["version"]

if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

# True when no gate failed. A run without rows has no failed gate.
checks_passed = stage_error is None and len(failed_checks) == 0

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                                  # audit_id
    run_id,                                             # run_id
    PIPELINE_NAME,                                      # pipeline_name
    GOLD_TABLE,                                         # gold_table
    processing_scope,                                   # processing_scope
    source_date_value,                                  # source_date
    affected_dates,                                     # affected_dates
    run_started_at,                                     # run_started_at
    run_finished_at,                                    # run_finished_at
    run_status,                                         # status
    error_message,                                      # error_message
    silver_row_counts,                                  # source_rows_read
    {"missing_event_date": rows_without_event_date},    # rows_excluded
    published_rows,                                     # gold_rows_published
    gold_table_version,                                 # gold_table_version
    checks_passed,                                      # checks_passed
    failed_checks_text,                                 # failed_checks
    quality_metrics,                                    # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)

if run_status == "NO_INPUT":
    print("WARNING: Silver has no rows in the scope of this run. The Gold table was not changed.")

if rows_without_event_date > 0:
    print("WARNING: rows without an event date that were left out:", rows_without_event_date)

if run_status == "FAILED":
    raise RuntimeError("repository_contributor_concentration FAILED: " + error_message)

## 11. Results

These cells only read. They show the audit row of this run and the Gold table as it is after this run.

### 11.1 The audit row of this run

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE run_id = '{run_id}'
      AND pipeline_name = '{PIPELINE_NAME}'
"""))

### 11.2 The Gold table per day

Expected events on the real week. Each day is the sum of the six Silver tables on that day: the events of G6 on that day without its watch, fork and release events.

| Day | Engineering events |
|---|---|
| 2024-02-01 | 4,649,982 |
| 2024-02-02 | 4,666,392 |
| 2024-02-03 | 4,220,907 |
| 2024-02-04 | 4,183,626 |
| 2024-02-05 | 4,515,629 |
| 2024-02-06 | 4,721,651 |
| 2024-02-07 | 4,799,212 |

The rows per day were not measured before. On every day, this table has at most as many rows as G6.

In [0]:
display(spark.sql(f"""
    SELECT
        activity_date,
        COUNT(*)                            AS repository_days,
        SUM(engineering_activity_events)    AS engineering_events,
        SUM(attributable_events)            AS attributable_events,
        SUM(unattributable_events)          AS unattributable_events,
        SUM(unique_active_actors)           AS actors,
        SUM(top_actor_events)               AS top_actor_events,
        SUM(top_3_actor_events)             AS top_3_actor_events,
        count_if(unique_active_actors = 1)  AS single_actor_rows
    FROM {GOLD_TABLE}
    GROUP BY activity_date
    ORDER BY activity_date
"""))

### 11.3 The whole Gold table in one row

Expected after a run for all dates on the real week, from the profiling run of 14 November 2026:

| Value | Expected |
|---|---|
| Rows | 4,318,172 |
| Engineering events | 31,757,399: the rows of the six Silver tables |
| Unattributable events | 0, so the coverage of the whole week is 100 |
| People (person-repository-days) | Not measured before. At least 833,632 (the people of G4, whose three tables are part of these six) and at most 6,881,398 (the people of G6, over all nine tables) |

The two shares in this cell are the shares of the whole week. They are calculated from the sums, not as the average of the shares of the rows. The top actor share of the week says how much of the work was done by the most active person of each repository-day.

In [0]:
display(spark.sql(f"""
    SELECT
        COUNT(*)                            AS repository_days,
        COUNT(DISTINCT activity_date)       AS days,
        SUM(engineering_activity_events)    AS engineering_events,
        SUM(attributable_events)            AS attributable_events,
        SUM(unattributable_events)          AS unattributable_events,
        SUM(unique_active_actors)           AS actors,
        SUM(top_actor_events)               AS top_actor_events,
        SUM(top_3_actor_events)             AS top_3_actor_events,
        count_if(attributable_events = 0)   AS rows_without_attributable_events,
        count_if(unique_active_actors = 1)  AS single_actor_rows,
        CASE
            WHEN SUM(engineering_activity_events) > 0
            THEN ROUND(100 * SUM(attributable_events) / SUM(engineering_activity_events), 2)
        END                                 AS actor_coverage_pct_of_all_rows,
        CASE
            WHEN SUM(attributable_events) > 0
            THEN ROUND(100 * SUM(top_actor_events) / SUM(attributable_events), 2)
        END                                 AS top_actor_share_pct_of_all_rows
    FROM {GOLD_TABLE}
"""))

### 11.4 How concentrated the rows are

The rows grouped by their number of people. A row with one person always has the top share 100 and the HHI 1, so this cell shows how many rows are concentrated only because one person was active.

In [0]:
display(spark.sql(f"""
    SELECT
        CASE
            WHEN unique_active_actors = 0  THEN '0 people'
            WHEN unique_active_actors = 1  THEN '1 person'
            WHEN unique_active_actors = 2  THEN '2 people'
            WHEN unique_active_actors = 3  THEN '3 people'
            WHEN unique_active_actors <= 10 THEN '4 to 10 people'
            ELSE 'more than 10 people'
        END                                   AS people_group,
        COUNT(*)                              AS repository_days,
        SUM(engineering_activity_events)      AS engineering_events,
        ROUND(AVG(top_actor_share_pct), 2)    AS average_top_actor_share_pct,
        ROUND(AVG(hhi_actor_activity), 4)     AS average_hhi
    FROM {GOLD_TABLE}
    GROUP BY people_group
    ORDER BY MIN(unique_active_actors)
"""))

### 11.5 A sample: the ten repository-days with the most people

A look at real rows. The repository-days with the most events are automated and have one or two people, so this sample sorts by people instead.

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {GOLD_TABLE}
    ORDER BY unique_active_actors DESC, repo_id, activity_date
    LIMIT 10
"""))

## What this notebook wrote

- `gold.repository_contributor_concentration`: the affected days were replaced, when the status is `SUCCEEDED`.
- `gold_staging.repository_contributor_concentration`: the staging copy, overwritten on every run.
- `audit.gold_pipeline_runs`: one new row.

With a `test_schema`, all three are in that schema.

## How to read the table

- A repository with one active person on a day has a top share of 100 and an HHI of 1 by definition. Compare only repository-days with enough events and people. 59% of the rows are built from one or two events.
- Bots are people in this table (decision 6). Logins that end in `[bot]` produce 17.5% of the engineering events of the week.
- The largest accounts are automation with an ordinary login. The single most active account produced 16.9% of all engineering events in only 30 repositories. A filter on `[bot]` would not remove them.
- One event is not one unit of work: a push with 100 commits is one event.
- Pushes are 83.6% of the engineering events, so the result mostly reflects who pushes.
- The share of several rows together is calculated from the sums. It is not the average of the shares of the rows.